# 07 — Truy vấn trên Kaggle, KHÔNG cần Elasticsearch/Milvus

Chạy cả tầng truy vấn online trên Kaggle bằng **bản gộp nạp thẳng vào RAM**.

## Vì sao notebook này tồn tại

10 trong 12 arm đọc Elasticsearch và Milvus. Cả hai là **máy chủ**, không phải
thư viện — compose gắn chúng vào `127.0.0.1`, và Kaggle không chạy Docker được.

Ở quy mô batch 1 thì không cần chúng:

* **Dense** — 176.707 vector × 1152 chiều đã chuẩn hoá L2, nên cosine **là một
  phép nhân ma trận**. numpy làm trong ~0,2 giây. Milvus nhanh hơn nhờ chỉ mục
  xấp xỉ; ở đây ta muốn *chính xác*, và 0,2 giây thì không ai chờ.
* **Văn bản** — `rank_bm25`, đúng thứ `InvertedIndex` đang dùng.

Tên arm, `Candidate` và `query_language` **giữ nguyên**, nên fusion, dedup và
DANTE không biết chúng đang chạy trên kho nào. Đó là điều kiện để so kết quả
Kaggle với kết quả local — đổi tên arm là mất luôn phép so.

## Hai Dataset phải gắn vào notebook

| Dataset | Nội dung | Bắt buộc |
|---|---|---|
| `mfe-btc-b1-merged` | `siglip_vectors.npz`, `subtitles.json`, `ocr_text.json` | có |
| `mfe-btc-queries` | `map-keyframes-aic25-b1.zip` + thư mục đề `query-*.txt` | có |

`audio_features.npz` **không cần** — tầng online chưa có arm CLAP, tải 329 MB
lên để nằm im.

## Cần bật

**Internet ON** (clone repo + tải model encoder). GPU chỉ đáng khi bật reranker:
đo trên CPU đã là 1–2 giây/truy vấn.

## Step 0 — cấu hình

In [ ]:
# ── ĐỔI HAI DÒNG NÀY CHO KHỚP TÊN DATASET BẠN UPLOAD ─────────────────────
MERGED_DS  = "/kaggle/input/mfe-btc-b1-merged"
QUERIES_DS = "/kaggle/input/mfe-btc-queries"

REPO_URL   = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF   = "feature/triet/structure"

CONFIG     = "config/btc_b1.yaml"
TOP_K      = 100      # = tran 100 dap an/truy van cua BTC (§2.2)
OUT        = "/kaggle/working/submission"

# Bo doc Q&A. "none" = de trong cot dap an, dien tay sau.
QA_READER  = "gemini"
QA_MODEL   = "gemini-3.6-flash"

import os, subprocess, sys, zipfile
from pathlib import Path

REPO = Path("/kaggle/working/mfe")
if not REPO.is_dir():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
    subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_REF, url, str(REPO)],
                   check=True, capture_output=True)   # token khong lot vao log
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "rank_bm25", "sentence-transformers", "google-genai"], check=True)

for name, path in (("ban gop", MERGED_DS), ("de + map", QUERIES_DS)):
    p = Path(path)
    print(f"{name:<10} {'CO' if p.is_dir() else 'KHONG THAY'}  {path}")
    if p.is_dir():
        for f in sorted(p.rglob("*"))[:12]:
            if f.is_file():
                print(f"    {f.relative_to(p)}  {f.stat().st_size/1e6:.1f} MB")


## Step 1 — dựng Relational Map cho 873 video

`candidate_frame_id` tra `frame_idx` từ bảng này. Không có nó thì lượt chạy chết
ở truy vấn đầu tiên.

Chỉ cần `map-keyframes` (~1,6 MB nén) — **không** cần 28 GB ảnh mà `mfe-ingest`
đòi. Xem docstring `scripts/build_btc_relational.py` để biết đã kiểm những gì.

In [ ]:
MAP_DIR = Path("/kaggle/working/data/btc_b1/map-keyframes")
MAP_DIR.mkdir(parents=True, exist_ok=True)

if not any(MAP_DIR.glob("*.csv")):
    zips = list(Path(QUERIES_DS).rglob("map-keyframes*.zip"))
    if zips:
        z = zipfile.ZipFile(zips[0])
        names = [n for n in z.namelist() if n.lower().endswith(".csv")]
        for n in names:
            # Giai nen PHANG: zip cua BTC boc trong mot thu muc, con config
            # tro thang vao {video}.csv — long thu muc la moi tep deu
            # "khong tim thay", va bang se rong ma khong ai keu.
            with z.open(n) as fh, (MAP_DIR / Path(n).name).open("wb") as out:
                out.write(fh.read())
        print(f"giai nen {len(names)} CSV tu {zips[0].name}")
    else:
        csvs = list(Path(QUERIES_DS).rglob("*.csv"))
        for f in csvs:
            (MAP_DIR / f.name).write_bytes(f.read_bytes())
        print(f"chep {len(csvs)} CSV")

print(f"map-keyframes: {len(list(MAP_DIR.glob('*.csv')))} tep")


In [ ]:
# media-info KHONG bat buoc cho truy van: no chi dien bang `videos`, ma
# `candidate_frame_id` doc bang `keyframes`. Thieu thi dung mac dinh, va
# GHI RO trong tep rang do la mac dinh — de sau nay khong ai doc no nhu that.
INFO_DIR = Path("/kaggle/working/data/btc_b1/media-info")
INFO_DIR.mkdir(parents=True, exist_ok=True)
found = list(Path(QUERIES_DS).rglob("media-info*.zip"))
if found:
    z = zipfile.ZipFile(found[0])
    for n in [x for x in z.namelist() if x.lower().endswith(".json")]:
        with z.open(n) as fh, (INFO_DIR / Path(n).name).open("wb") as out:
            out.write(fh.read())
    print(f"media-info that: {len(list(INFO_DIR.glob('*.json')))} tep")
else:
    import json as _json
    for csv_path in MAP_DIR.glob("*.csv"):
        (INFO_DIR / f"{csv_path.stem}.json").write_text(
            _json.dumps({"_nguon": "mac dinh — media-info khong duoc upload"}),
            encoding="utf-8")
    print(f"KHONG co media-info -> mac dinh cho {len(list(INFO_DIR.glob('*.json')))} video")


In [ ]:
import json
import yaml
cfg = yaml.safe_load(Path(CONFIG).read_text(encoding="utf-8"))
cfg["paths"]["map_keyframes"] = str(MAP_DIR)
cfg["paths"]["media_info"]    = str(INFO_DIR)
cfg["paths"]["relational_db"] = "/kaggle/working/stores/btc_b1/relational/app.db"
cfg["paths"]["eval_results"]  = "/kaggle/working/eval/results"
cfg["paths"]["throughput"]    = "/kaggle/working/eval/throughput.json"
cfg["paths"]["event_log"]     = "/kaggle/working/eval/events.jsonl"
# Cache dich VI->EN: uu tien ban co san trong dataset.
#
# `siglip` la arm manh nhat va thap text cua no CHI hieu tieng Anh, nen
# khong co ban dich thi no ngoi ngoai — mat arm tot nhat ma khong loi nao
# no ra. Tren Kaggle 21-08-2026, envit5 khong nap duoc (`tier mt failed
# (0)`), va no thu lai o TUNG truy van: 1,2 GB tai lai moi lan, hong moi
# lan. Dich san o may khac roi gui cache len la duong ngan nhat, va no
# dung DUNG model + DUNG tham so greedy, nen ket qua khong khac gi.
cached = list(Path(QUERIES_DS).rglob("translations.json"))
CACHE = Path("/kaggle/working/eval/translations.json")
CACHE.parent.mkdir(parents=True, exist_ok=True)
if cached:
    CACHE.write_bytes(cached[0].read_bytes())
    n = len(json.loads(CACHE.read_text(encoding="utf-8")).get("entries", []))
    print(f"cache dich: {n} muc tu {cached[0].name}")
else:
    print("KHONG co translations.json trong dataset -> se tai envit5 (1,2 GB).")
    print("Neu no hong, arm `siglip` se ngoi ngoai — kiem bang arm o Step 3.")
cfg["retrieval"]["translation"]["cache_path"] = str(CACHE)

KCFG = Path("/kaggle/working/btc_b1_kaggle.yaml")
KCFG.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(f"cau hinh cho Kaggle: {KCFG}")

subprocess.run([sys.executable, "scripts/build_btc_relational.py",
                "--config", str(KCFG)], check=True)


## Step 2 — nạp bản gộp vào RAM

`siglip_vectors.npz` là ~765 MB; nạp mất khoảng một phút rồi nằm trong RAM suốt
phiên. BM25 dựng **lười** — chỉ nguồn nào thật sự bị truy vấn mới dựng, vì dựng
đủ cả bảy nguồn tốn hàng phút cho những arm có khi không dùng tới.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")
for noisy in ("httpx", "httpcore", "urllib3", "filelock"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

from mfe.online.flat_arms import FlatBundle

bundle = FlatBundle.load(MERGED_DS)
print("\n" + bundle.report())


## Step 3 — dựng cascade, và ĐỌC KỸ bảng arm

Bảng này nói arm nào **khả dụng**, không nói arm nào **có dữ liệu**.
`ocr_paddle`/`ocr_vietocr` sẽ `unavailable` nếu bạn không upload hai tệp đó —
đúng như mong đợi khi Trọng chưa chạy xong, và tốt hơn là chúng im lặng trả rỗng.

`clip` và `bm25` cũng ngồi ngoài: chúng đọc kho local (Faiss + inverted index)
mà notebook này không dựng. 10 arm còn lại là toàn bộ sức mạnh của hệ.

In [ ]:
from mfe.core.config import load_config
from mfe.mapping.relational import RelationalMap
from mfe.online.cascade import Cascade

config = load_config(str(KCFG))
relational = RelationalMap(config.paths.relational_db)
relational.init_schema()
print(f"relational: {relational.count_keyframes()} keyframe")

def _no_milvus(collection, dimension):
    raise RuntimeError("khong dung Milvus — dang chay tren ban gop")

cascade = Cascade.from_config(
    config,
    flat_bundle=bundle,
    vector_index=None,
    inverted_index=None,
    text_store=None,
    dense_store_factory=_no_milvus,
)

rows = list(cascade.arms)
on = [a for a in rows if a.active]
print(f"\narm: {len(on)}/{len(rows)} chay duoc")
for arm in rows:
    mark = "  ON " if arm.active else "  --- "
    why = "" if arm.active else f"  ({arm.status()})"
    print(f"{mark}{arm.name:<18}{why}")
tr = type(cascade.translator).__name__ if cascade.translator else "None <- siglip se ngoi ngoai"
print(f"\ntranslator: {tr}")


## Step 4 — chạy cả gói đề, ghi file nộp

In [ ]:
import json as _json
from mfe.eval.query_package import load_package
from mfe.eval import submission as submission_io
from mfe.online import answering

sys.path.insert(0, str(REPO / "scripts"))
from run_submission import PLACEHOLDER, run_query

qdirs = [p for p in Path(QUERIES_DS).rglob("*")
         if p.is_dir() and any(p.glob("query-*.txt"))]
QUERY_DIR = qdirs[0] if qdirs else Path(QUERIES_DS)
queries = load_package(QUERY_DIR)
print(f"de: {QUERY_DIR} — {len(queries)} truy van")

reader = images = None
if QA_READER == "gemini" and any(q.type == "QA" for q in queries):
    if os.environ.get("GEMINI_API_KEY"):
        from mfe.online.readers import GeminiAnswerReader
        reader = GeminiAnswerReader(model=QA_MODEL, images=3)
        # Khong co anh keyframe tren Kaggle -> bo doc chi dung van ban.
        images = answering.KeyframeImages(relational)
    else:
        print(f"  ! khong co GEMINI_API_KEY -> cot dap an se la {PLACEHOLDER!r}")

submit_dir = Path(OUT) / "submission"
submit_dir.mkdir(parents=True, exist_ok=True)
notes = []
for i, q in enumerate(queries, 1):
    rows_out, note = run_query(q, cascade, relational, TOP_K,
                               reader=reader, images=images, n_images=3)
    path = submission_io.write_csv(rows_out, submit_dir / q.csv_name, limit=TOP_K)
    note["rows"] = len(rows_out); note["file"] = path.name
    notes.append(note)
    print(f"[{i:>2}/{len(queries)}] {q.query_id:<22} {q.type:<5} {len(rows_out):>3} dong")

Path(OUT).mkdir(parents=True, exist_ok=True)
(Path(OUT) / "run-report.json").write_text(
    _json.dumps({"arms_on": [a.name for a in cascade.arms if a.active],
                 "queries": notes}, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nda ghi {len(notes)} tep -> {submit_dir}")


In [ ]:
import zipfile as _zip
archive = Path(OUT) / "aic2026.zip"
with _zip.ZipFile(archive, "w", _zip.ZIP_DEFLATED) as z:
    for f in sorted(submit_dir.glob("*.csv")):
        z.write(f, arcname=f"submission/{f.name}")
print(f"{archive}  ({archive.stat().st_size/1e3:.0f} KB)")

no_answer = [n for n in notes if n.get("answer_source") in
             ("MODEL BO CUOC", "BO DOC HONG", "KHONG CO BO DOC")]
if no_answer:
    print(f"\nCHUA CO DAP AN ({len(no_answer)} cau Q&A) — cot dap an la {PLACEHOLDER!r}:")
    for n in no_answer:
        print(f"  {n['query_id']}  [{n['answer_source']}]")
